# 17 · Diagnostics on saved models: where the open-set failures come from (Stage R7)

## Purpose
Three earlier stages pointed at three possible mechanisms that were never tested directly. This stage tests each with the
**saved** models only — **no model is trained** — and decides whether any further training experiment is worth running
before code freeze (CLAUDE.md §7, Stage R7, declared before computing anything; development data only; reserved partitions
sealed; nothing here is an independent claim; the bundle, rules and thresholds are unchanged).

- **D1 — the mild horizontal misalignment blind spot (R6).** The healthy-only autoencoder misses horizontal misalignment at
  0.5 mm. Does the classifier gate, which decides under the default rule, catch these recordings?
- **D2 — feature collapse (R4).** Does a CNN trained without a class discard the physical directions that separate it (for
  withheld vertical misalignment: the axial / radial and 2× / 1× order ratios), so the withheld class lands in a known
  cluster? Tested with a linear probe from the embedding to fixed physics descriptors.
- **D3 — a saturated probability gate.** Known-side top probabilities sit near 1, above τ_p ≈ 0.9975. Would a logit-based
  confidence term (max-logit or energy), with the same Mahalanobis term, reject more unfamiliar recordings at the same
  known-side cost?

## Inputs
Saved LOCO CNN-6 runs (`artifacts/loco/cnn6_w-<c>_s<seed>_loco/fold<k>.pt` and their saved window logits, embeddings and
recording scores in `results/scores/`), the Q3 assignment (`results/q1q3/q3_fit_cal_eval_assignment.csv`), seed-42 thresholds
(`results/loco/decision_thresholds.csv`), B1s (`results/scores/b1s_nomic_loco_q3fit_scores.parquet`), the capped R1 feature
table (`ml.cache.load_features`), the closed-set context and the reproduction record from `scripts/37_r7_forward.py`
(`results/research/r7_closed_set_hm.csv`, `r7_reproduction.json`), and R6's per-recording scores
(`results/research/r6_per_recording.csv`). Config and split hashes are printed below.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> loads the configuration, metadata and development pool, prints the config and split hashes, and reads the reproduction record: every one of the 81 saved runs (9 withheld classes × 3 folds × 3 seeds) was run forward again and must match its saved outputs before anything below is used. It stops if any run failed.</div>

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from ml.arbitrator import decide
from ml.cache import load_features
from ml.config import load_config, config_sha256
from ml.diagnostics import DESCRIPTORS, fit_probe, gate_rejects, logit_scores, matched_u_threshold, probe_scores, recording_embeddings
from ml.labels import CLASS10
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR, data_dir
from ml.protocol import order_statistic_threshold
from ml.q1 import mild_mask
from ml.rejection import TIE_EPS, clopper_pearson, snap_ties
from ml.splits import FAULT_CLASSES, development_pool, split_hash

ROOT = SYSTEM_CODE_DIR
R = ROOT / "results"
SC = R / "scores"
cfg = load_config(DEFAULT_CONFIG)
meta = pd.read_csv(ROOT / "metadata.csv")
print("config sha256:", config_sha256(cfg)[:16], "| split hash:", split_hash(meta)[:16])
pool = development_pool(meta, cfg.development.allowed_closed, cfg.development.reserved_columns).merge(
    pd.read_csv(R / cfg.development.folds_file)[["rec_id", "dev_fold"]], on="rec_id")
info = meta.set_index("rec_id")
SEEDS = (42, 43, 44)
rep = json.loads((R / "research/r7_reproduction.json").read_text())
rr = pd.DataFrame(rep["runs"])
assert rep["all_reproduced"] and len(rr) == 81, "a saved output did not reproduce: stop and report"
print(f"reproduced {int(rr.reproduced.sum())} of {len(rr)} runs; max |Δ logit| {rr.max_abs_logit.max():.1e}, max |Δ embedding| "
      f"{rr.max_abs_embedding.max():.1e}, max |Δ MSP| {rr.max_abs_msp.max():.1e}, max relative Δ Mahalanobis {rr.max_rel_maha.max():.1e}")
print("seed-42 thresholds recomputed:", rep["seed42_thresholds"]); print("note:", rep["note"])

## The saved scores and thresholds of every run
Per run and seed: the recording scores (top class, MSP, Mahalanobis, AE-6 error) and the run's thresholds, recomputed from its
calibration recordings with the existing rules (τ_p, τ_d: the 97.5 % order statistic; τ_e: the maximum calibration-normal AE
error; B1s and τ_b are seed-independent). Seed 42's must equal the saved threshold file.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> loads the recording scores of all 81 runs, computes each run's four thresholds from its calibration recordings, checks seed 42 against the saved threshold file, and attaches B1s and τ_b.</div>

In [ ]:
b1s = pd.read_parquet(SC / "b1s_nomic_loco_q3fit_scores.parquet")[["withheld", "outer_fold", "rec_id", "b1s", "tau_b"]]
thr42 = pd.read_csv(R / "loco/decision_thresholds.csv").set_index(["withheld", "outer_fold"])
recs, thr = [], []
for s in SEEDS:
    for c in FAULT_CLASSES:
        for k in (0, 1, 2):
            x = pd.read_parquet(SC / f"cnn6_w-{c}_s{s}_loco_fold{k}_recording.parquet")
            cal = x[x.role == "cal"]
            t = {"seed": s, "withheld": c, "outer_fold": k,
                 "tau_p": -order_statistic_threshold(-cal.msp.to_numpy(), 0.975)["threshold"],
                 "tau_d": order_statistic_threshold(cal.maha.to_numpy(), 0.975)["threshold"],
                 "tau_e": float(cal[cal.class10 == "normal"].ae.max())}
            if s == 42:
                assert abs(t["tau_p"] - thr42.loc[(c, k), "tau_p"]) <= 1e-12 and abs(t["tau_d"] - thr42.loc[(c, k), "tau_d"]) <= 1e-12
                assert abs(t["tau_e"] - thr42.loc[(c, k), "tau_e"]) <= 1e-12
            thr.append(t)
            recs.append(x)
THR = pd.DataFrame(thr)
REC = pd.concat(recs, ignore_index=True).merge(b1s, on=["withheld", "outer_fold", "rec_id"], how="left") \
    .merge(THR, on=["seed", "withheld", "outer_fold"])
assert REC.b1s.notna().all()
print(REC.groupby("seed").size().to_dict(), "| thresholds per seed:", THR.groupby("seed").size().to_dict())

## D1 — does the classifier gate catch mild horizontal misalignment?
Population: the 25 development-pool horizontal misalignment 0.5 mm recordings (all in one outer fold, because folds are
folder-grouped), with the 14 that R6's A0 missed at seed 42 marked. **Gate verdict (Bhuvesh, 1 Oct):** the closed-set
development CNN-6 has no valid calibration, so the default rule `ml.arbitrator.decide("B_with_caution")` is applied with the
saved LOCO runs that **learned** horizontal misalignment (withheld ≠ horizontal misalignment: 8 runs per seed), for which these
recordings are out of sample (role foldk_known), each with its own thresholds. A recording counts as **called healthy** on a
seed if at least 4 of its 8 runs say healthy.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> collects, for every 0.5 mm recording and seed, the verdicts of the 8 runs that learned horizontal misalignment (healthy, horizontal misalignment, another fault, further inspection), counts the recordings called healthy per seed (majority and strict 'any run'), marks R6's A0 misses, and applies the declared reading.</div>

In [ ]:
hm = pool[(pool.class10 == "horizontal_misalignment") & np.isclose(pool.severity, 0.5)].rec_id
r6 = pd.read_csv(R / "research/r6_per_recording.csv")
a0 = r6[(r6.variant == "A0") & (r6.seed == 42)]
from ml.speed_calibration import exceeds, matched_threshold
t0 = matched_threshold(a0[~a0.fault].z.to_numpy(), int(exceeds(a0[~a0.fault].z.to_numpy(), 0.0).sum()))
a0_missed = set(a0[a0.fault & ~exceeds(a0.z.to_numpy(), t0)].rec_id) & set(hm)
d = snap_ties(REC[REC.rec_id.isin(hm) & (REC.withheld != "horizontal_misalignment") & (REC.role == "foldk_known")].copy())
def verdict(r):
    o = decide("B_with_caution", r.pred, float(r.msp), float(r.maha), float(r.ae), float(r.b1s),
               {"tau_p": r.tau_p, "tau_d": r.tau_d, "tau_e": r.tau_e, "tau_b": r.tau_b})
    if o["decision"] == "healthy":
        return "healthy"
    if o["decision"] == "unknown":
        return "further inspection"
    return "horizontal misalignment" if o["fault_class"] == "horizontal_misalignment" else "another fault"
d["verdict"] = [verdict(r) for r in d.itertuples()]
assert (d.role == "foldk_known").all() and (d.groupby(["seed", "rec_id"]).size() == 8).all()
V = d.groupby(["seed", "rec_id"]).verdict.value_counts().unstack(fill_value=0)
V["called_healthy_majority"] = V.get("healthy", 0) >= 4
V["called_healthy_any"] = V.get("healthy", 0) >= 1
V["A0_missed"] = [r in a0_missed for r in V.index.get_level_values("rec_id")]
D1 = V.groupby("seed").agg(called_healthy_majority=("called_healthy_majority", "sum"), called_healthy_any=("called_healthy_any", "sum"))
tot = d.groupby("seed").verdict.value_counts().unstack(fill_value=0)
print("verdicts over the 25 recordings × 8 runs (200 per seed):"); print(tot.to_string())
print(D1.to_string()); print(f"R6 A0 misses among these recordings at seed 42: {len(a0_missed)}")
mh = D1.called_healthy_majority
D1_READING = ("covered by the gate" if (mh <= 1).all() else "system blind spot" if (mh >= 3).any() else "unclear")
print("D1 reading:", D1_READING)

### Threshold-free context: the closed-set development CNN-6 on the same recordings
Top class, top probability, the probability given to vertical misalignment (the confusable pair) and the closed-set Mahalanobis
distance (no threshold exists for these models), from the forward-pass script; top classes match the saved Stage 3A / 3B
predictions.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> summarises, per seed, the closed-set CNN-6's top classes on the 25 recordings, how often the top class is vertical misalignment, the lowest top probability, and the distance range; then draws the LOCO verdict counts per recording (seed 42) with R6's A0 misses marked, and saves the figure.</div>

In [ ]:
cs = pd.read_csv(R / "research/r7_closed_set_hm.csv")
assert cs.top_matches_saved.all() and set(cs.rec_id) == set(hm)
print(cs.groupby("seed").agg(top_hm=("top_class", lambda v: int((v == "horizontal_misalignment").sum())),
                             top_vm=("top_class", lambda v: int((v == "vertical_misalignment").sum())),
                             top_other=("top_class", lambda v: int((~v.isin(["horizontal_misalignment", "vertical_misalignment"])).sum())),
                             min_top_probability=("top_probability", "min"), max_p_vm=("p_vertical_misalignment", "max"),
                             maha_min=("maha_closed_set", "min"), maha_max=("maha_closed_set", "max")).round(4).to_string())
print("LOCO runs: vertical misalignment named for", int((d.pred == "vertical_misalignment").sum()), "of", len(d), "run-recording pairs")
v42 = V.loc[42]
cols = [c for c in ("healthy", "horizontal misalignment", "another fault", "further inspection") if c in v42]
colors = {"healthy": "#2ca02c", "horizontal misalignment": "#4f6bff", "another fault": "#d62728", "further inspection": "#ff8c00"}
fig, ax = plt.subplots(figsize=(12, 4.2))
bottom = np.zeros(len(v42))
labels = [f"{r.split('__')[-1]} Hz" + (" *" if v42.loc[r, "A0_missed"] else "") for r in v42.index]
for c in cols:
    ax.bar(range(len(v42)), v42[c], bottom=bottom, color=colors[c], label=c); bottom += v42[c].to_numpy()
ax.set_xticks(range(len(v42)), labels, rotation=60, ha="right", fontsize=7)
ax.set_ylabel("runs (of 8) giving this verdict"); ax.axhline(4, ls=":", color="k", lw=1)
ax.set_title("Horizontal misalignment 0.5 mm: default-rule verdicts of the 8 LOCO runs that learned it (seed 42; * = missed by R6's A0)", fontsize=10)
ax.legend(fontsize=8, loc="lower right"); fig.tight_layout(); fig.savefig(R / "figures/r7_d1_mild_hm.png", dpi=150); plt.show()

## D2 — is there feature collapse? A linear probe from the embedding to physics descriptors
For every run and seed: recording embeddings (the mean of the 9 saved, reproduced window embeddings); the 16 descriptors fixed
in the declaration (recording value = mean of the 9 windows of the capped R1 table), both standardised with the run's **fit**
recordings (known classes only); a ridge probe fitted on those fit recordings with α by 5-fold CV on them; R² per descriptor on
known evaluation recordings (roles eval and foldk_known) and on withheld recordings (foldk_withheld). The probe never sees a
withheld recording. The descriptor list, written before fitting:

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints the fixed descriptor list, loads the capped R1 feature table and averages each descriptor over each recording's nine windows.</div>

In [ ]:
print(DESCRIPTORS)
fpath = max((data_dir() / "features").glob(f"{cfg.baselines.feature_table}_*.parquet"), key=lambda p: p.stat().st_mtime)
F = load_features(fpath, meta, cfg.preprocess)
DESC = F[F.rec_id.isin(pool.rec_id)].groupby("rec_id")[DESCRIPTORS].mean()
assert len(DESC) == len(pool)
print("descriptor table:", DESC.shape, "| feature file:", fpath.name)

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> fits the probe for each of the 81 runs on its fit recordings only, scores R² and RMSE per descriptor on the known evaluation and the withheld recordings, and records each run's withheld-class Mahalanobis AUROC (unseen-fold known vs withheld).</div>

In [ ]:
rows = []
for s in SEEDS:
    for c in FAULT_CLASSES:
        for k in (0, 1, 2):
            E = np.load(SC / f"cnn6_w-{c}_s{s}_loco_fold{k}_embeddings.npz")
            Z = recording_embeddings(E["embedding"], E["rec_id"])
            role = pd.Series(E["role"], index=E["rec_id"]).groupby(level=0).first()
            fit_ids = role.index[role == "fit"]
            assert not (info.loc[fit_ids].class10 == c).any()                      # leakage guard: no withheld recording
            kn_ids = role.index[role.isin(["eval", "foldk_known"])]
            wh_ids = role.index[role == "foldk_withheld"]
            p = fit_probe(Z.loc[fit_ids].to_numpy(), DESC.loc[fit_ids].to_numpy(), seed=s)
            a = probe_scores(p, Z.loc[kn_ids].to_numpy(), DESC.loc[kn_ids].to_numpy())
            b = probe_scores(p, Z.loc[wh_ids].to_numpy(), DESC.loc[wh_ids].to_numpy())
            x = REC[(REC.seed == s) & (REC.withheld == c) & (REC.outer_fold == k) & REC.role.isin(["foldk_known", "foldk_withheld"])]
            rows.append({"seed": s, "withheld": c, "outer_fold": k, "alpha": p["alpha"], "n_fit": len(fit_ids),
                         "n_known_eval": len(kn_ids), "n_withheld": len(wh_ids),
                         "r2_known": a["r2_mean"], "r2_withheld": b["r2_mean"], "rmse_known": a["rmse_mean"], "rmse_withheld": b["rmse_mean"],
                         "maha_auroc": roc_auc_score(x.role == "foldk_withheld", x.maha),
                         **{f"r2_known__{n}": v for n, v in zip(DESCRIPTORS, a["r2"])},
                         **{f"r2_withheld__{n}": v for n, v in zip(DESCRIPTORS, b["r2"])}})
P = pd.DataFrame(rows)
cls = P.groupby(["withheld", "seed"])[["r2_known", "r2_withheld", "rmse_known", "rmse_withheld", "maha_auroc"]].mean()
print(cls.round(3).to_string())

### D2 reading
Collapse is supported only if, for withheld vertical misalignment, mean withheld R² < 0.5 while known R² ≥ 0.8 on all three
seeds, and across each seed's 27 runs the withheld R² correlates with the Mahalanobis AUROC (Spearman > 0.4). It is not
supported if withheld R² ≥ 0.8 for vertical misalignment on all seeds.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> applies the declared reading: vertical misalignment's known and withheld R² per seed, the per-seed Spearman correlation of withheld R² with Mahalanobis AUROC over the 27 runs, and the per-descriptor R² for vertical misalignment. Then, as a supplement labelled post hoc, the same correlation with the probe's RMSE, and a figure of the withheld RMSE against AUROC and of known vs withheld RMSE per class (R² on one withheld class is measured against that class's own narrow variance, so it is not interpretable as a plot), saved for the report.</div>

In [ ]:
vm = P[P.withheld == "vertical_misalignment"].groupby("seed")[["r2_known", "r2_withheld"]].mean()
rho = {s: spearmanr(P[P.seed == s].r2_withheld, P[P.seed == s].maha_auroc)[0] for s in SEEDS}
print(vm.round(3).to_string()); print("Spearman(withheld R², Mahalanobis AUROC) per seed:", {k: round(v, 3) for k, v in rho.items()})
supported = bool(((vm.r2_withheld < 0.5) & (vm.r2_known >= 0.8)).all() and all(v > 0.4 for v in rho.values()))
not_supported = bool((vm.r2_withheld >= 0.8).all())
D2_READING = "collapse supported" if supported else ("collapse not supported" if not_supported else "unclear")
print("D2 reading:", D2_READING)
pd_vm = P[P.withheld == "vertical_misalignment"]
print(pd.DataFrame({"known": [pd_vm[f"r2_known__{n}"].mean() for n in DESCRIPTORS],
                    "withheld": [pd_vm[f"r2_withheld__{n}"].mean() for n in DESCRIPTORS]}, index=DESCRIPTORS).round(3).to_string())
# supplementary (post hoc, not part of the declared reading): R² on a single withheld class is computed against that
# class's own, narrow variance and is not interpretable; RMSE in fit-standardised units compares like with like
rho_rmse = {s: spearmanr(P[P.seed == s].rmse_withheld, P[P.seed == s].maha_auroc)[0] for s in SEEDS}
print("supplementary, post hoc: Spearman(withheld RMSE, Mahalanobis AUROC) per seed:", {k: round(v, 3) for k, v in rho_rmse.items()})
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for c, col in zip(FAULT_CLASSES, plt.cm.tab10.colors):
    q = P[P.withheld == c]
    axes[0].scatter(q.rmse_withheld, q.maha_auroc, color=col, s=22, label=c.replace("_", " "))
axes[0].set_xlabel("probe RMSE on withheld recordings (fit-standardised units)"); axes[0].set_ylabel("Mahalanobis AUROC (withheld vs known)")
axes[0].set_title("Probe error on the withheld class vs familiarity AUROC, 81 runs", fontsize=10); axes[0].legend(fontsize=7)
g = P.groupby("withheld")[["rmse_known", "rmse_withheld"]].mean().loc[list(FAULT_CLASSES)]
xx = np.arange(len(g))
axes[1].bar(xx - 0.2, g.rmse_known, 0.4, label="known evaluation", color="#9e9e9e")
axes[1].bar(xx + 0.2, g.rmse_withheld, 0.4, label="withheld", color="#4f6bff")
axes[1].set_xticks(xx, [c.replace("_", " ") for c in g.index], rotation=35, ha="right", fontsize=8)
axes[1].set_ylabel("probe RMSE (fit-standardised units)"); axes[1].set_title("Probe error by withheld class (mean over seeds and folds)", fontsize=10)
axes[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(R / "figures/r7_d2_probe.png", dpi=150); plt.show()

## D3 — is the probability half of the gate saturated? Logit-based gates
Per recording: max-logit (mean over the 9 windows of the window's maximum logit) and energy (mean over the 9 windows of
−logsumexp); unknown scores u1 = −max-logit, u2 = energy, thresholded at the 97.5 % order statistic of the run's calibration
recordings exactly as τ_p. Gates with the unchanged Mahalanobis term: G0 (MSP ≥ τ_p), G1 (u1 ≤ τ1), G2 (u2 ≤ τ2). Primary: unknown
recall on the withheld recordings at a matched known-side rejection count (G1 / G2's logit threshold moved, τ_d fixed, to reject
exactly as many unseen-fold known recordings as G0).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> computes the two logit scores from every run's saved (reproduced) window logits, the run thresholds, and for each gate and seed: known-side rejections, unknown recall at G0's rejection count and at the calibrated thresholds, withheld recordings called healthy, and the saturation of the known side (top probability above 0.999).</div>

In [ ]:
g_rows, sat = [], []
for s in SEEDS:
    for c in FAULT_CLASSES:
        known = [q for q in CLASS10 if q != c]
        for k in (0, 1, 2):
            L = pd.read_parquet(SC / f"cnn6_w-{c}_s{s}_loco_fold{k}_logits.parquet")
            U = logit_scores(L[[f"logit_{q}" for q in known]].to_numpy(), L.rec_id.to_numpy())
            x = REC[(REC.seed == s) & (REC.withheld == c) & (REC.outer_fold == k)].set_index("rec_id").join(U)
            cal = x[x.role == "cal"]
            t1 = order_statistic_threshold(cal.u_maxlogit.to_numpy(), 0.975)["threshold"]
            t2 = order_statistic_threshold(cal.u_energy.to_numpy(), 0.975)["threshold"]
            kn, un = x[x.role == "foldk_known"], x[x.role == "foldk_withheld"]
            tau_d, tau_p = float(x.tau_d.iloc[0]), float(x.tau_p.iloc[0])
            g0_kn = gate_rejects(-kn.msp, -tau_p, kn.maha, tau_d)
            g0_un = gate_rejects(-un.msp, -tau_p, un.maha, tau_d)
            n0 = int(g0_kn.sum())
            sat.append({"seed": s, "withheld": c, "outer_fold": k, "n_known": len(kn) + len(cal),
                        "above_0999": int((pd.concat([kn, cal]).msp > 0.999).sum())})
            for gname, u, t in (("G0", -x.msp, -tau_p), ("G1", x.u_maxlogit, t1), ("G2", x.u_energy, t2)):
                uk, uu = u.loc[kn.index], u.loc[un.index]
                tm = t if gname == "G0" else matched_u_threshold(uk.to_numpy(), kn.maha.to_numpy(), tau_d, n0)
                rej_cal = gate_rejects(uu, t, un.maha, tau_d)
                rej_m = gate_rejects(uu, tm, un.maha, tau_d) if tm is not None else np.full(len(un), np.nan)
                healthy = (~rej_cal) & (un.pred == "normal").to_numpy()
                healthy_m = (~rej_m.astype(bool)) & (un.pred == "normal").to_numpy() if tm is not None else np.full(len(un), np.nan)
                for rid, rc, rm, hc, hm_ in zip(un.index, rej_cal, rej_m, healthy, healthy_m):
                    g_rows.append({"seed": s, "withheld": c, "outer_fold": k, "gate": gname, "rec_id": rid, "rejected_calibrated": bool(rc),
                                   "rejected_matched": rm, "healthy_calibrated": bool(hc), "healthy_matched": hm_,
                                   "matched_reachable": tm is not None, "known_rejected_calibrated": int(gate_rejects(uk, t, kn.maha, tau_d).sum()),
                                   "known_rejected_G0": n0})
G = pd.DataFrame(g_rows)
SAT = pd.DataFrame(sat)
print("unreachable matched points:", int((~G.drop_duplicates(["seed", "withheld", "outer_fold", "gate"]).matched_reachable).sum()))
print("known side with top probability > 0.999:", {s: f"{q.above_0999.sum()}/{q.n_known.sum()} = {q.above_0999.sum() / q.n_known.sum():.3f}" for s, q in SAT.groupby("seed")})
summ = G.groupby(["seed", "gate"]).agg(unknown_recall_matched=("rejected_matched", "mean"), unknown_recall_calibrated=("rejected_calibrated", "mean"),
                                        withheld_healthy_matched=("healthy_matched", "sum"), withheld_healthy_calibrated=("healthy_calibrated", "sum"))
kr = G.drop_duplicates(["seed", "withheld", "outer_fold", "gate"]).groupby(["seed", "gate"])[["known_rejected_calibrated", "known_rejected_G0"]].sum()
print(summ.join(kr).round(3).to_string())

### D3 reading, per withheld class and at the mildest settings
G1 / G2 helps only if matched unknown recall is at least 5 points above G0 on all three seeds with withheld recordings called
healthy not higher on any seed; does not help if the seed-mean difference is ≤ 0 or healthy calls rise on any seed; otherwise
too thin to tell.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> applies the declared reading to G1 and G2, prints matched unknown recall per withheld class and at the mildest setting of each family (seed mean), and draws the per-class comparison, saved for the report.</div>

In [ ]:
D3_READING = {}
for gname in ("G1", "G2"):
    dlt = {s: float(summ.loc[(s, gname), "unknown_recall_matched"] - summ.loc[(s, "G0"), "unknown_recall_matched"]) for s in SEEDS}
    hup = {s: int(summ.loc[(s, gname), "withheld_healthy_matched"]) > int(summ.loc[(s, "G0"), "withheld_healthy_matched"]) for s in SEEDS}
    helps = all(v >= 0.05 for v in dlt.values()) and not any(hup.values())
    nohelp = np.mean(list(dlt.values())) <= 0 or any(hup.values())
    D3_READING[gname] = {"delta_per_seed": dlt, "healthy_higher": hup,
                         "reading": "helps" if helps else ("does not help" if nohelp else "too thin to tell")}
    print(gname, {k: round(v, 4) for k, v in dlt.items()}, "healthy higher:", hup, "→", D3_READING[gname]["reading"])
pc = G.groupby(["withheld", "gate"]).rejected_matched.mean().unstack()[["G0", "G1", "G2"]]
print(pc.round(3).to_string())
un_info = info.loc[G.rec_id.unique()][["class10", "fault_family", "severity"]]
mild = mild_mask(un_info, cfg.anomaly)
mrows = [{"setting": lab, "gate": gname, "n": int(m.sum()),
          "recall_matched": float(G[(G.gate == gname) & G.rec_id.isin(m[m].index)].rejected_matched.mean())}
         for lab, m in mild.items() for gname in ("G0", "G1", "G2")]
print(pd.DataFrame(mrows).pivot_table(index="setting", columns="gate", values="recall_matched").round(3).to_string())
fig, ax = plt.subplots(figsize=(11, 4.4))
xx = np.arange(len(pc))
for j, (gname, col) in enumerate((("G0", "#9e9e9e"), ("G1", "#4f6bff"), ("G2", "#ff8c00"))):
    ax.bar(xx + (j - 1) * 0.27, pc[gname], 0.27, color=col, label={"G0": "G0 probability", "G1": "G1 max-logit", "G2": "G2 energy"}[gname])
ax.set_xticks(xx, [c.replace("_", " ") for c in pc.index], rotation=30, ha="right", fontsize=8)
ax.set_ylabel("withheld called unknown (matched known rejections)"); ax.set_ylim(0, 1.05); ax.legend(fontsize=8)
ax.set_title("Unknown recall per withheld class at G0's known-side rejection count (seeds 42/43/44 pooled)", fontsize=10)
fig.tight_layout(); fig.savefig(R / "figures/r7_d3_gates.png", dpi=150); plt.show()

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> writes the results: the JSON summary (reproduction, the three readings with their numbers, saturation) and the three companion tables (D1 per recording and seed, D2 per run, D3 per run, gate and withheld recording).</div>

In [ ]:
out = {"stage": "R7", "seeds": list(SEEDS), "split_hash": split_hash(meta), "config_sha256": config_sha256(cfg),
       "reproduction": {"all_reproduced": rep["all_reproduced"], "runs": len(rr), "note": rep["note"],
                        "max_rel_maha": float(rr.max_rel_maha.max()), "max_abs_msp": float(rr.max_abs_msp.max())},
       "d1": {"reading": D1_READING, "called_healthy_majority": D1.called_healthy_majority.to_dict(),
              "called_healthy_any": D1.called_healthy_any.to_dict(), "verdict_totals": tot.to_dict("index"),
              "a0_missed_seed42": sorted(a0_missed),
              "closed_set_top_vm": cs.groupby("seed").top_class.apply(lambda v: int((v == "vertical_misalignment").sum())).to_dict()},
       "d2": {"reading": D2_READING, "vm_per_seed": vm.to_dict("index"), "spearman_r2_auroc": rho,
              "supplementary_post_hoc_spearman_rmse_auroc": rho_rmse,
              "per_class": P.groupby("withheld")[["r2_known", "r2_withheld", "maha_auroc"]].mean().to_dict("index")},
       "d3": {"reading": D3_READING, "summary": summ.reset_index().to_dict("records"),
              "saturation_known_above_0999": {s: float(q.above_0999.sum() / q.n_known.sum()) for s, q in SAT.groupby("seed")},
              "per_class_matched": pc.to_dict("index")}}
(R / "research/r7_diagnostics.json").write_text(json.dumps(out, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
V.reset_index().to_csv(R / "research/r7_d1_mild_hm.csv", index=False)
P.to_csv(R / "research/r7_d2_probe.csv", index=False)
G.to_csv(R / "research/r7_d3_gates.csv", index=False)
print("written: r7_diagnostics.json, r7_d1_mild_hm.csv, r7_d2_probe.csv, r7_d3_gates.csv")

## Findings

Exploratory diagnostics on saved models (no training), development data, seeds 42 / 43 / 44. **All 81 saved LOCO runs
reproduce exactly** (window logits, embeddings, recording MSP and Mahalanobis: maximum difference 0; seed-42 thresholds
within 1.1e-16). A first reproduction pass cast the embeddings to float64 before the Ledoit–Wolf fit and differed by up to
2.7e-4; Stage 3C fitted on float32 embeddings, and with that path everything matches (recorded in `r7_reproduction.json`).

- **D1 — covered by the gate.** For the 25 horizontal misalignment 0.5 mm recordings, the 8 LOCO runs per seed that learned
  the class call **none** of them healthy, on any seed and by any run (0 of 200 verdicts per seed): 174–190 of 200 name
  horizontal misalignment, 10–26 send to further inspection, at most 1 names another fault. This includes all 14 recordings
  R6's healthy-only A0 missed at seed 42. The closed-set development CNN-6 tops all 25 with horizontal misalignment on every
  seed (lowest top probability 0.84 at seed 43), and neither model ever names vertical misalignment for them. **The R6 blind
  spot is a property of the healthy-only detector; the classifier-first default rule already handles it.**
- **D2 — formally "unclear"; the evidence does not point to collapse.** The declared R² statistic is degenerate on a single
  withheld class (it is measured against that class's own narrow variance: mean withheld R² −18 to −2,814), and even on known
  evaluation recordings the probe reaches only R² −0.22 to 0.43 by class (vertical misalignment withheld: 0.27 / 0.30 / 0.34), far from the
  0.8 the reading requires; Spearman(withheld R², AUROC) is −0.22 / −0.35 / −0.47. **Supplementary, post hoc:** in
  fit-standardised RMSE the probe predicts the withheld vertical misalignment recordings' descriptors almost as well as known
  ones (1.00 vs 0.91; imbalance 1.23 vs 0.93), while the bearing classes are far off the known map (overhang ball 2.67), and
  Spearman(withheld RMSE, AUROC) is +0.34 / +0.55 / +0.24. So the physics directions are not discarded for the vertical
  misalignment / imbalance pair: those withheld recordings sit where the known-class map predicts them, which is why
  Mahalanobis cannot tell them apart; the 16 descriptors are only weakly (linearly) encoded for every class.
- **D3 — the probability half is saturated, but swapping it is too thin to tell.** 68–71 % of known recordings have top
  probability > 0.999 (τ_p per run 0.60–0.998). At G0's known-side rejection count, max-logit (G1) and energy (G2) raise unknown recall by
  **+1.2 to +1.4 points** on every seed (G0 0.612 / 0.633 / 0.634), with no extra withheld recordings called healthy (G1 15 / 17
  / 24 vs G0 19 / 17 / 24). The gain is concentrated where Mahalanobis is weak: withheld vertical misalignment 0.160 → 0.192,
  overhang cage 0.778 → 0.824–0.833, overhang cage 0 g 0.68 → 0.79; imbalance stays at 0.05. Below the 5-point bar.


## What this stage decides

**Readings:** D1 covered by the gate · D2 unclear (collapse not supported by the evidence) · D3 too thin to tell (G1, G2).

**Decision by the declared rules: no further training experiment before code freeze.** The auxiliary physics-descriptor
head (Stage R8) was to be declared only if D2 supported collapse, which it does not: the withheld vertical misalignment
recordings are predicted almost as well as known ones, so the failure is not a lost direction but that the withheld class
sits where the known-class map puts it. The logit-based gates move unknown recall by little more than one point, far below
the 5-point bar, so they are reported as a development observation only; the deployed gate is unchanged. The remaining
open-set gap — withheld imbalance (unknown recall ≈ 0.05) and vertical misalignment (≈ 0.16–0.19) — is documented, not
chased. D1 goes into the write-up beside R6: mild horizontal misalignment is missed by the healthy-only autoencoder but is
named correctly by the classifier gate in 87–95 % of run verdicts and never called healthy.


## Outputs written
- `results/research/r7_diagnostics.json`, `r7_d1_mild_hm.csv`, `r7_d2_probe.csv`, `r7_d3_gates.csv` (this notebook);
  `results/research/r7_reproduction.json`, `r7_closed_set_hm.csv` (`scripts/37_r7_forward.py`, forward passes only).
- Figures: `results/figures/r7_d1_mild_hm.png`, `r7_d2_probe.png`, `r7_d3_gates.png`.